# Week 18, Day 1: Hypothesis testing: null hypotheses, p-values and errors

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w018/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A z-test from scratch

Write `z_test(x, sigma, mu0)` returning the z statistic (x̄ − μ0)/(σ/√n) and the two-sided p-value 2(1 − Φ(|z|)), with Φ from `math.erf`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt
def Phi(z): return 0.5 * (1 + erf(z / sqrt(2)))

def z_test(x, sigma, mu0):
    z = ...
    p = ...
    return z, p

print(z_test(np.array([1.0, 2.0, 3.0, 4.0]), 2.0, 1.0))

In [ ]:
import numpy as np
def test_values():
    "z_test returns the z statistic and the two-sided p-value"
    z, p = z_test(np.array([1.0, 2.0, 3.0, 4.0]), 2.0, 1.0)
    assert abs(z - 1.5) < 1e-12 and abs(p - 0.13361440253771617) < 1e-9

def test_symmetric():
    "the p-value depends only on the size of z"
    assert abs(z_test(np.array([-1.0, -2.0]), 1.0, 0.0)[1] - z_test(np.array([1.0, 2.0]), 1.0, 0.0)[1]) < 1e-12

test_values()
test_symmetric()
print("All checks passed")

### Exercise: The distribution of p-values

Simulate 10,000 experiments of 20 Gaussian observations with σ = 1, testing μ0 = 0. Store p-values when the true mean is 0 in `p_null` and when it is 0.4 in `p_alt`, and plot both histograms.

In [ ]:
rng = np.random.default_rng(2016)
p_null = np.array([...])
p_alt = np.array([...])
print(f"fraction below 0.05: null {np.mean(p_null < 0.05):.3f}, real effect {np.mean(p_alt < 0.05):.3f}")
plt.hist(p_null, 20, alpha=0.5, label="null true"); plt.hist(p_alt, 20, alpha=0.5, label="effect 0.4"); plt.legend(); plt.xlabel("p-value"); plt.show()

In [ ]:
import numpy as np
def test_null_uniform():
    "under the null about 5% of p-values fall below 0.05, and each tenth of [0, 1] holds about 10%"
    assert abs(np.mean(p_null < 0.05) - 0.05) < 0.01
    assert np.all(np.abs(np.histogram(p_null, bins=10, range=(0, 1))[0] / len(p_null) - 0.1) < 0.015)

def test_alt():
    "under a real effect more than 40% of p-values fall below 0.05"
    assert np.mean(p_alt < 0.05) > 0.4

test_null_uniform()
test_alt()
print("All checks passed")

### Exercise: A paired test on spike counts

On 40 trials, a neuron's baseline count varies with a large slow drift, and the stimulus adds about 1.5 spikes. Compute the paired t statistic `t_stat` and its two-sided p-value `p_paired` (Gaussian approximation), and compare with `p_unpaired` from treating baseline and response as independent groups: z = (mean difference) / sqrt(var_b / n + var_r / n).

In [ ]:
rng_p = np.random.default_rng(2)
n = 40
drift = 10 + 8 * np.sin(np.linspace(0, 3 * np.pi, n))       # slow changes in excitability
base = rng_p.poisson(drift)
resp = rng_p.poisson(drift + 1.5)
d = resp - base
t_stat = ...
p_paired = 2 * (1 - Phi(abs(t_stat)))
z_unp = ...
p_unpaired = 2 * (1 - Phi(abs(z_unp)))
print(f"paired: t = {t_stat:.2f}, p = {p_paired:.4f};  unpaired: z = {z_unp:.2f}, p = {p_unpaired:.4f}")

In [ ]:
import numpy as np
def test_t():
    "t is the mean difference over its standard error"
    d = resp - base
    assert abs(t_stat - d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))) < 1e-12

def test_paired_wins():
    "the paired test gives a smaller p-value than an unpaired test that ignores the pairing"
    assert p_paired < p_unpaired

test_t()
test_paired_wins()
print("All checks passed")

### Exercise: How often is significant real?

Simulate 100,000 studies: each tests a real effect with probability 0.1 and has power 0.5 (so a real effect is significant with probability 0.5, a null one with probability 0.05). Compute the fraction of significant studies that tested a real effect (`ppv_sim`) and compare with the formula (`ppv_formula`).

In [ ]:
N = 100000
real = rng.random(N) < 0.1
significant = np.where(real, rng.random(N) < 0.5, rng.random(N) < 0.05)
ppv_sim = ...
ppv_formula = ...
print(f"P(real | significant): simulated {ppv_sim:.3f}, formula {ppv_formula:.3f}")

In [ ]:
import numpy as np
def test_sim():
    "the simulated PPV matches the formula to within 0.03"
    assert abs(ppv_sim - ppv_formula) < 0.03

def test_low():
    "with 10% real effects and 50% power, fewer than 60% of significant results are real"
    assert ppv_formula < 0.6

test_sim()
test_low()
print("All checks passed")